In [23]:
# %pip install folium # Creating a map

In [24]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import folium

In [25]:
# create data = mumbai -> 19.0814342,72.7135411
np.random.seed(42) # same result every time

In [26]:
# cluster 1 # bandra (80 points)
lat1 = 19.06+np.random.normal(0,0.015,80)
lon1 = 72.83+np.random.normal(0,0.015,80)

# cluster 2 # andheri (70 points)
lat2 = 19.11+np.random.normal(0,0.012,70)
lon2 = 72.87+np.random.normal(0,0.012,70)

# cluster 3 # colaba (60 point)
lat3 = 18.92+np.random.normal(0,0.008,60)
lon3 = 72.82+np.random.normal(0,0.008,60)

# noise points (40 random scattered)
lat_noise = np.random.uniform(18.8,19.4,40)
lon_noise = np.random.uniform(72.7,73.1,40)


In [27]:
# all combine 
lats = np.concatenate([lat1,lat2,lat3,lat_noise])
lons = np.concatenate([lon1,lon2,lon3,lon_noise])

In [28]:
# final Array : shape
coords = np.column_stack((lats,lons))

In [29]:
coords_rad = np.radians(coords)

In [30]:
print("total GPS point = ",len(coords))

total GPS point =  250


In [31]:
# set DBSCAN perameters set  
kms_per_radian = 6371.0
epsilon_km = 1.2 # (0.5 to 2 try)
epsilon_red = kms_per_radian / epsilon_km

In [32]:
min_sample = 5 # 3 to 10 best result 

In [33]:
# add value into model 
db = DBSCAN(eps=epsilon_red,
            min_samples=min_sample,
            algorithm="ball_tree", # GPS fast
            metric="haversine").fit(coords_rad)

In [34]:
labels = db.labels_ # number of clusters (-1 noise)

In [35]:
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"cluster {n_clusters}")
print(f"Noise / outliner points = {n_noise}")

cluster 1
Noise / outliner points = 0


In [36]:
df = pd.DataFrame({'latitude':lats,'longitude':lons,'cluster':labels})
print(df.head())

    latitude  longitude  cluster
0  19.067451  72.826705        0
1  19.057926  72.835357        0
2  19.069715  72.852168        0
3  19.082845  72.822226        0
4  19.056488  72.817873        0


In [ ]:
# map visualization (Folium)
m = folium.Map(location=[19.0760,72.8777] , zoom_start=11,tiles="CartoDB positron")

colors = ['blue','green','purple','orange','darkred']

for idx , row  in df.iterrows():
    if row['cluster'] == -1:
        color = "red"
        size = 4
        text = "Noise / outlier"
    else:
        color = colors[row['cluster'] % len(colors)]
        size = 6
        text = f"Cluster {row["cluster"]}"  
        
    folium.CircleMarker(
        location= [row['latitude'],row['longitude']],
        radius=size,
        color= color,
        fill = True,
        fill_opacity=0.7,
        tooltip=text
    ).add_to(m)     
    
m.save("mumbai.html")     

TypeError: list indices must be integers or slices, not numpy.float64